# 02 — Cálculo de indicadores

Lee los archivos de `datos/filtrado/` y genera los 4 CSV de análisis en `datos/analisis/`.

La columna `fac_tri` ya viene normalizada desde `01_filtrado` — todos los trimestres
usan el mismo nombre sin importar el año de origen.

In [1]:
import pandas as pd
import numpy as np
import re
import gc
from pathlib import Path

RAIZ          = Path('..').resolve()
RUTA_FILTRADO = RAIZ / 'datos' / 'filtrado'
RUTA_SALIDA   = RAIZ / 'datos' / 'analisis'

RUTA_SALIDA.mkdir(parents=True, exist_ok=True)

archivos = sorted(RUTA_FILTRADO.glob('ENOE_T*.csv'))
print(f'Filtrados encontrados: {len(archivos)}')
if len(archivos) == 0:
    raise RuntimeError('No hay filtrados. Corre primero 01_filtrado.ipynb')

Filtrados encontrados: 80


## 1. Unir todos los trimestres

In [2]:
def parsear_nombre(ruta):
    n = Path(ruta).stem.upper()
    for pref in ['ENOE_TSDEMT','ENOEN_TSDEMT','ENOE_T','ENOEN_T']:
        if n.startswith(pref):
            suf = n[len(pref):]
            if re.fullmatch(r'\d{3,4}', suf):
                t, a = int(suf[0]), int('20'+suf[1:])
                if t in [1,2,3,4] and 2000 <= a <= 2030:
                    return t, a
    return None


trozos = []
for ruta in archivos:
    info = parsear_nombre(ruta)
    if not info:
        continue
    t, a = info
    df = pd.read_csv(ruta, encoding='utf-8-sig', low_memory=False)
    df['trimestre'] = t
    df['anio']      = a
    df['periodo']   = f'{a}-T{t}'
    trozos.append(df)
    del df; gc.collect()

datos = pd.concat(trozos, ignore_index=True)
del trozos; gc.collect()

print(f'Registros: {len(datos):,}  |  Años: {datos["anio"].min()}–{datos["anio"].max()}')

n_fac = int(datos['fac_tri'].notna().sum()) if 'fac_tri' in datos.columns else 0
print(f'Con fac_tri válido: {n_fac:,} ({n_fac/len(datos)*100:.1f}%)')

Registros: 13,813,820  |  Años: 2005–2025
Con fac_tri válido: 13,813,820 (100.0%)


## 2. Ingreso por hora

In [3]:
for col in ['ingocup','hrsocup','ing_x_hrs','ingreso_hora']:
    if col in datos.columns:
        datos[col] = pd.to_numeric(datos[col], errors='coerce')

# ingreso_hora ya viene calculado desde 01_filtrado
# Este paso rellena huecos residuales por si algún archivo fue procesado diferente
if 'ingreso_hora' not in datos.columns:
    datos['ingreso_hora'] = np.nan

if 'ingocup' in datos.columns and 'hrsocup' in datos.columns:
    i, h = datos['ingocup'], datos['hrsocup']
    datos['ingreso_hora'] = datos['ingreso_hora'].fillna(
        (i / (h * 4.33)).round(2).where((i > 0) & (h > 0))
    )

datos['ingreso_hora'] = datos['ingreso_hora'].where(datos['ingreso_hora'] > 0)
print(f'ingreso_hora válido: {datos["ingreso_hora"].notna().sum():,}')

ingreso_hora válido: 9,779,640


## 3. Mediana ponderada con `fac_tri`

Tanto la mediana como la media usan `fac_tri` para representar a la **población total**
y no solo a la muestra.

In [7]:
def mediana_ponderada(valores, pesos=None):
    v = pd.to_numeric(valores, errors='coerce')
    mask = v.notna()
    if pesos is not None:
        w = pd.to_numeric(pesos, errors='coerce')
        mask = mask & w.notna() & (w > 0)
    if mask.sum() == 0:
        return np.nan
    v_arr = v[mask].values
    if pesos is None or mask.sum() < 30:
        return float(np.median(v_arr))
    w_arr = w[mask].values
    orden = np.argsort(v_arr)
    cum   = np.cumsum(w_arr[orden])
    idx   = min(np.searchsorted(cum, cum[-1] / 2), len(v_arr) - 1)
    return float(round(v_arr[orden][idx], 2))


def agrupar(df, by, val, peso='fac_tri'):
    tiene_w = peso in df.columns
    rows = []
    for keys, g in df.groupby(by):
        med = mediana_ponderada(g[val], g[peso] if tiene_w else None)
        if not isinstance(keys, tuple):
            keys = (keys,)
        rows.append(dict(zip(by, keys)) | {'mediana_ing_hora': med})
    return pd.DataFrame(rows)


print('Función de mediana ponderada lista.')

Función de mediana ponderada lista.


In [8]:
def weighted_mean(x, w):
    """Media ponderada con fac_tri."""
    mask = x.notna() & w.notna() & (w > 0)
    if mask.sum() == 0:
        return np.nan
    return float((x[mask] * w[mask]).sum() / w[mask].sum())


def mediana_ponderada(valores, pesos=None):
    """Mediana ponderada: el valor donde la suma acumulada de fac_tri supera el 50%."""
    v = pd.to_numeric(valores, errors='coerce')
    mask = v.notna()
    if pesos is not None:
        w = pd.to_numeric(pesos, errors='coerce')
        mask = mask & w.notna() & (w > 0)
    if mask.sum() == 0:
        return np.nan
    v_arr = v[mask].values
    if pesos is None or mask.sum() < 30:
        return float(np.median(v_arr))
    w_arr = w[mask].values
    orden = np.argsort(v_arr)
    cum   = np.cumsum(w_arr[orden])
    idx   = min(np.searchsorted(cum, cum[-1]/2), len(v_arr)-1)
    return float(round(v_arr[orden][idx], 2))


def agrupar(df, by, val, peso='fac_tri'):
    tiene_w = peso in df.columns
    rows = []
    for keys, g in df.groupby(by):
        med = mediana_ponderada(g[val], g[peso] if tiene_w else None)
        if not isinstance(keys, tuple): keys = (keys,)
        rows.append(dict(zip(by, keys)) | {'mediana_ing_hora': med})
    return pd.DataFrame(rows)


print('Funciones ponderadas listas.')

Funciones ponderadas listas.


## 4. Serie temporal nacional

In [9]:
d = datos[datos['ingreso_hora'].notna() & datos['sexo'].notna()]
serie = agrupar(d, ['anio','sexo'], 'ingreso_hora').sort_values(['anio','sexo'])
# Serie nacional (solo por año)
serie_nac = agrupar(d, ['anio'], 'ingreso_hora').rename(columns={'mediana_ing_hora':'mediana_nacional'})
# Unir ambas tablas
serie = serie.merge(serie_nac, on='anio', how='left')
serie.to_csv(RUTA_SALIDA / 'serie_nacional.csv', index=False, encoding='utf-8-sig')
print('serie_nacional.csv guardado.')
print(serie.tail(6).to_string(index=False))

serie_nacional.csv guardado.
 anio   sexo  mediana_ing_hora  mediana_nacional
 2023 Hombre             40.00             38.76
 2023  Mujer             37.50             38.76
 2024 Hombre             44.44             43.65
 2024  Mujer             42.00             43.65
 2025 Hombre             50.00             48.45
 2025  Mujer             46.51             48.45


## 5. Brecha salarial por sector y año

In [10]:
d2 = datos[
    datos['ingreso_hora'].notna() &
    datos['sexo'].notna() &
    datos['sector'].notna()
]
bs = agrupar(d2, ['anio','sector','sexo'], 'ingreso_hora')
bp = bs.pivot_table(
    index=['anio','sector'], columns='sexo', values='mediana_ing_hora'
).reset_index()
bp.columns.name = None
if 'Hombre' in bp.columns and 'Mujer' in bp.columns:
    bp['brecha_pp']  = (bp['Hombre'] - bp['Mujer']).round(2)
    bp['brecha_pct'] = (bp['brecha_pp'] / bp['Hombre'] * 100).round(1)
bp.to_csv(RUTA_SALIDA / 'brecha_sector_anio.csv', index=False, encoding='utf-8-sig')
print('brecha_sector_anio.csv guardado.')
print(bp[bp['anio'] == datos['anio'].max()].sort_values('brecha_pct', ascending=False).to_string(index=False))

brecha_sector_anio.csv guardado.
 anio                                sector  Hombre  Mujer  brecha_pp  brecha_pct
 2025                              Comercio   43.75  37.50       6.25        14.3
 2025                           Manufactura   52.08  44.68       7.40        14.2
 2025              Restaurantes y hospedaje   48.45  41.67       6.78        14.0
 2025                    Servicios diversos   52.08  46.15       5.93        11.4
 2025                    Servicios sociales   93.02  83.72       9.30        10.0
 2025 Servicios financieros y profesionales   58.33  58.33       0.00         0.0
 2025                          Agropecuario   35.19  37.50      -2.31        -6.6
 2025                              Gobierno   65.20  69.77      -4.57        -7.0
 2025                          Construcción   52.17  62.50     -10.33       -19.8
 2025          Transportes y comunicaciones   50.00  60.29     -10.29       -20.6
 2025                  Industria extractiva   62.50  81.40     -1

## 6. Brecha por nivel educativo y sector

In [11]:
ult = int(datos['anio'].max())
d3  = datos[
    (datos['anio'] == ult) &
    datos['ingreso_hora'].notna() &
    datos['sexo'].notna() &
    datos['nivel_educ'].notna() &
    datos['sector'].notna()
]
be  = agrupar(d3, ['sector','nivel_educ','sexo'], 'ingreso_hora')
bep = be.pivot_table(
    index=['sector','nivel_educ'], columns='sexo', values='mediana_ing_hora'
).reset_index()
bep.columns.name = None
if 'Hombre' in bep.columns and 'Mujer' in bep.columns:
    bep['brecha_pct'] = ((bep['Hombre'] - bep['Mujer']) / bep['Hombre'] * 100).round(1)
bep.to_csv(RUTA_SALIDA / f'brecha_educ_sector_{ult}.csv', index=False, encoding='utf-8-sig')
print(f'brecha_educ_sector_{ult}.csv guardado.')

brecha_educ_sector_2025.csv guardado.


## 7. Informalidad como contexto de apoyo

In [12]:
if 'formalidad' in datos.columns:
    d4 = datos[
        (datos['anio'] == ult) &
        datos['sector'].notna() &
        datos['sexo'].notna() &
        datos['formalidad'].notna()
    ]
    tiene_fac = 'fac_tri' in d4.columns
    rows = []
    for (sec, sex, form), g in d4.groupby(['sector','sexo','formalidad']):
        # Población expandida: suma de fac_tri en lugar de conteo simple
        n = float(g['fac_tri'].sum()) if tiene_fac else len(g)
        rows.append({'sector':sec,'sexo':sex,'formalidad':form,'n':n})
    inf = pd.DataFrame(rows)
    tot = inf.groupby(['sector','sexo'])['n'].sum().reset_index(name='total')
    inf = inf.merge(tot, on=['sector','sexo'])
    inf['pct_informalidad'] = (inf['n'] / inf['total'] * 100).round(1)
    inf = inf[inf['formalidad'] == 'Informal'].drop(columns=['formalidad','n','total'])
    inf.to_csv(RUTA_SALIDA / f'informalidad_contexto_{ult}.csv', index=False, encoding='utf-8-sig')
    print(f'informalidad_contexto_{ult}.csv guardado.')

print('\nArchivos generados en datos/analisis/:')
for f in sorted(RUTA_SALIDA.glob('*.csv')):
    kb    = f.stat().st_size / 1024
    filas = sum(1 for _ in open(f)) - 1
    print(f'  {f.name:<45}  {filas:>5} filas  {kb:.0f} KB')

informalidad_contexto_2025.csv guardado.

Archivos generados en datos/analisis/:
  brecha_educ_sector_2020.csv                       55 filas  3 KB
  brecha_educ_sector_2025.csv                       55 filas  3 KB
  brecha_estructural_2025.csv                      383 filas  29 KB
  brecha_ingreso_sector_ocupacion_2025.csv         502 filas  39 KB
  brecha_ocupacion_anio.csv                        231 filas  13 KB
  brecha_sector_anio.csv                           231 filas  11 KB
  brecha_sector_ocupacion_2025.csv               17588 filas  1521 KB
  informalidad_contexto_2020.csv                    22 filas  1 KB
  informalidad_contexto_2025.csv                    22 filas  1 KB
  serie_nacional.csv                                42 filas  1 KB


---
## 8. Tabla extendida: brecha por sector × ocupación × educación

Este bloque responde la pregunta más profunda del proyecto:
¿persiste la brecha cuando hombres y mujeres tienen el **mismo cargo y la misma escolaridad**?

Si un directivo hombre y una directiva mujer con estudios superiores en Manufactura
ganan diferente, ya no puede atribuirse ni al puesto ni a la formación —
eso es la brecha estructural en su forma más pura.

La tabla incluye `poblacion_ocupada` (suma de fac_tri) para saber cuántas personas
representa cada celda y descartar combinaciones con muestras muy pequeñas.

In [13]:
# Filtrar registros válidos: con ingreso_hora y factor de expansión
ocupados = datos[
    datos['ingreso_hora'].notna() &
    datos['fac_tri'].notna() &
    (datos['fac_tri'] > 0) &
    datos['sector'].notna() &
    datos['ocupacion'].notna() &
    datos['nivel_educ'].notna() &
    datos['sexo'].notna()
]

print(f'Registros válidos para tabla extendida: {len(ocupados):,}')

# Agrupar por año, sector, tipo de ocupación, nivel educativo y sexo
# Calcular: población representada, media ponderada y mediana ponderada del ingreso/hora
filas = []
for keys, g in ocupados.groupby(['anio','sector','ocupacion','nivel_educ','sexo']):
    anio_k, sector_k, ocu_k, educ_k, sexo_k = keys
    filas.append({
        'anio'             : anio_k,
        'sector'           : sector_k,
        'ocupacion'        : ocu_k,
        'nivel_educ'       : educ_k,
        'sexo'             : sexo_k,
        'poblacion_ocupada': round(g['fac_tri'].sum()),
        'media_ing_hora'   : round(weighted_mean(g['ingreso_hora'], g['fac_tri']), 2),
        'mediana_ing_hora' : mediana_ponderada(g['ingreso_hora'], g['fac_tri']),
    })

df_ocup = pd.DataFrame(filas)

archivo_ext = f'brecha_sector_ocupacion_{ult}.csv'
df_ocup.to_csv(RUTA_SALIDA / archivo_ext, index=False, encoding='utf-8-sig')
print(f'{archivo_ext} guardado  ({len(df_ocup):,} filas)')
print(f'Combinaciones únicas de sector×ocupación×educación: '
      f'{df_ocup[["sector","ocupacion","nivel_educ"]].drop_duplicates().shape[0]}')

Registros válidos para tabla extendida: 9,750,424
brecha_sector_ocupacion_2025.csv guardado  (17,516 filas)
Combinaciones únicas de sector×ocupación×educación: 576


## 9. Vista previa: brecha estructural (mismo cargo, misma escolaridad)

Pivotear para ver directamente cuánto ganan hombres vs mujeres
en el mismo grupo de ocupación y nivel educativo, dentro de cada sector.

In [14]:
# Solo el último año
df_ult = df_ocup[df_ocup['anio'] == ult].copy()

pivot_ext = df_ult.pivot_table(
    index=['sector','ocupacion','nivel_educ'],
    columns='sexo',
    values='mediana_ing_hora'
).reset_index()
pivot_ext.columns.name = None

if 'Hombre' in pivot_ext.columns and 'Mujer' in pivot_ext.columns:
    pivot_ext['brecha_pct'] = (
        (pivot_ext['Hombre'] - pivot_ext['Mujer']) / pivot_ext['Hombre'] * 100
    ).round(1)
    # Filtrar celdas con ambos sexos representados
    pivot_ext = pivot_ext.dropna(subset=['Hombre','Mujer'])

# Top 10 brechas más altas donde hombres ganan más
top10 = pivot_ext.nlargest(10, 'brecha_pct')[[
    'sector','ocupacion','nivel_educ','Hombre','Mujer','brecha_pct'
]]
print(f'Top 10 brechas estructurales más altas en {ult}:')
print(top10.to_string(index=False))
print()

# Guardar versión pivoteada
pivot_ext.to_csv(
    RUTA_SALIDA / f'brecha_estructural_{ult}.csv',
    index=False, encoding='utf-8-sig'
)
print(f'brecha_estructural_{ult}.csv guardado.')

Top 10 brechas estructurales más altas en 2025:
                      sector                    ocupacion      nivel_educ     Hombre     Mujer  brecha_pct
          Servicios diversos Trabajadores de la educación Sin instrucción 103.333335 18.518520        82.1
                    Comercio Trabajadores de la educación Sin instrucción 106.978175 35.685485        66.6
          Servicios diversos            Otras ocupaciones  Media superior  51.679590 17.857140        65.4
          Servicios diversos Trabajadores de la educación          Básica  60.000000 22.093020        63.2
          Servicios sociales        Trabajadores del arte No especificado 155.038760 59.011625        61.9
                    Gobierno Trabajadores de la educación Sin instrucción  67.441860 25.839790        61.7
                    Gobierno         Servicios personales          Básica  58.140000 23.255810        60.0
Transportes y comunicaciones            Otras ocupaciones  Media superior  77.097505 31.250000  